# Full-Corpus QLoRA Fine-Tuning: Scaling Qwen 2.5 Across webis/args_me

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dustinober1/Arg_Me/blob/main/notebooks/13_qwen_qlora_full_corpus_training.ipynb)

### Overview & Motivation
In Notebook 12, our 10-epoch pilot run demonstrated **spectacular convergence**:
- **Training Loss**: `0.0223` | **Validation Loss**: `0.0175` | **Token Accuracy**: `99.13%`
- **Dialectical Trap Resolution**: 100% correct resolution of both **Adversarial Quoting** (*Nuclear Energy Expansion*) and **Concession Complexity** (*Universal Basic Income*).

This notebook scales the parameter-efficient **QLoRA (4-bit NF4)** training across the **entire `webis/args_me` corpus** (~387,000 arguments from Hugging Face) combined with our mined hard-negative curriculum:
1. **Direct Cloud Ingestion**: Streams arguments on-the-fly directly from Hugging Face Hub (`webis/args_me`), requiring zero local JSONL downloading.
2. **4-Bit NF4 Quantization**: Loads `Qwen/Qwen2.5-3B-Instruct` in ~2.8 GB VRAM, fitting easily within a free Colab T4 GPU (15 GB) with headroom for large batch processing.
3. **Modern TRL SFTTrainer Pipeline**: Uses `SFTConfig` and `processing_class=tokenizer` with 8-bit Paged AdamW and cosine scheduling.
4. **Continuous Google Drive Checkpointing**: Persists adapter checkpoints automatically so long-running multi-epoch runs are never lost.

In [ ]:
# 1. Environment Detection, Auto-Cloning & GPU Verification
import sys
import os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    print("==> Provisioning environment inside Google Colab GPU runtime...")
    !pip install -q datasets huggingface_hub pandas pyarrow scikit-learn transformers accelerate bitsandbytes peft trl matplotlib seaborn tqdm
    if not os.path.exists("Arg_Me") and not os.path.exists("src"):
        print("==> Cloning Arg_Me repository into Colab...")
        !git clone https://github.com/dustinober1/Arg_Me.git
        %cd Arg_Me
    elif os.path.exists("Arg_Me") and not os.path.exists("src"):
        %cd Arg_Me
    if "/content/Arg_Me" not in sys.path:
        sys.path.insert(0, "/content/Arg_Me")
    print("==> Configured Colab working directory:", os.getcwd())
else:
    print("==> Running in local environment:", sys.executable)

import torch
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Notice: No CUDA GPU detected. (Training is optimized for CUDA / Colab T4/A100).")

## 2. Model Architecture & 4-bit NF4 Quantization Setup
We use **`Qwen/Qwen2.5-3B-Instruct`** (or `Qwen/Qwen2.5-1.5B-Instruct` / `Qwen2.5-7B-Instruct`).
4-bit NormalFloat (NF4) quantization + double quantization keeps base model memory at **~2.8 GB VRAM**.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = os.environ.get("QWEN_MODEL_ID", "Qwen/Qwen2.5-3B-Instruct")
print(f"Target Qwen Base Model: {MODEL_ID}")

# 4-Bit NormalFloat Quantization Configuration
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print(f"Downloading tokenizer for {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Downloading and loading 4-bit quantized base model {MODEL_ID}...")
device_map = "auto" if torch.cuda.is_available() else None
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map=device_map,
    trust_remote_code=True,
)

if torch.cuda.is_available():
    base_model = prepare_model_for_kbit_training(base_model)

# Configure LoRA Adapter Architecture (target all projection layers)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

## 3. Large-Scale / Full-Corpus Ingestion from Hugging Face Hub
We ingest debate arguments directly from `webis/args_me` on Hugging Face Hub.
- **Target Sample Limit**: Set `MAX_ARGUMENTS = None` for the entire ~387k dataset, or specify an integer (e.g. `10000`, `25000`, `50000`) for custom run lengths.
- **Curriculum Integration**: Automatically integrates the 32 mined hard negatives from active learning.

In [ ]:
import pandas as pd
from pathlib import Path
from datasets import Dataset
from sklearn.model_selection import train_test_split
from src.dataset_loader import load_full_args_corpus
from src.qwen_qlora_trainer import build_training_dataset, format_chat_record, prepare_hf_conversations

try:
    from src.argument_hygiene import clean_argument_dataframe
except ImportError:
    def clean_argument_dataframe(df):
        return df, pd.DataFrame()

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# ==========================================================================
# CONFIGURATION: Target Corpus Size
# Set MAX_ARGUMENTS = None to process the entire args.me corpus!
# Or set to e.g. 5000, 10000, 25000 for faster scaled training runs.
# ==========================================================================
MAX_ARGUMENTS = 10000  # Change to None for ALL ~387,000 arguments
CACHE_PARQUET = DATA_DIR / f"args_me_{MAX_ARGUMENTS or 'full'}.parquet"

if CACHE_PARQUET.exists():
    print(f"Loading cached corpus from: {CACHE_PARQUET}")
    df_raw = pd.read_parquet(CACHE_PARQUET)
else:
    # Check if local 5k benchmark exists as fallback
    fallback_path = DATA_DIR / "args_me_benchmark_5k.parquet"
    if fallback_path.exists() and (MAX_ARGUMENTS is not None and MAX_ARGUMENTS <= 5000):
        print(f"Loading existing local benchmark from {fallback_path}...")
        df_raw = pd.read_parquet(fallback_path)
    else:
        print("Streaming arguments directly from Hugging Face Hub (webis/args_me)...")
        df_raw = load_full_args_corpus(max_arguments=MAX_ARGUMENTS, save_parquet_path=str(CACHE_PARQUET))

# Clean procedural noise
df_clean, noise_df = clean_argument_dataframe(df_raw)
print(f"Loaded {len(df_clean):,} clean arguments (pruned {len(noise_df)} procedural chatter items).")

# 2. Load Mined Hard-Negative Curriculum
hn_path = DATA_DIR / "hard_negatives_curriculum.parquet"
df_hard = pd.read_parquet(hn_path) if hn_path.exists() else None
if df_hard is not None:
    print(f"Loaded {len(df_hard)} mined hard negatives from active learning curriculum!")
else:
    print("Mined hard negatives not found on disk, continuing with primary dataset.")

# 3. Build Stratified Training and Validation Splits
train_df, test_df = build_training_dataset(
    df=df_clean,
    hard_negatives_df=df_hard,
    hard_negative_weight=3,  # Boost hard negatives 3x
    test_size=0.10,
    random_state=42,
)

print(f"\nFinal Train Set Size: {len(train_df):,} | Test Set Size: {len(test_df):,}")
print("Train Stance Balance:", train_df["stance"].str.upper().value_counts().to_dict())

# 4. Format into Hugging Face Datasets with Qwen Chat Template
def apply_chat_template(examples):
    formatted_texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        formatted_texts.append(text)
    return {"text": formatted_texts}

print("Applying chat template to train and evaluation datasets...")
train_dataset = Dataset.from_list(prepare_hf_conversations(train_df)).map(apply_chat_template, batched=True)
eval_dataset = Dataset.from_list(prepare_hf_conversations(test_df.iloc[:250])).map(apply_chat_template, batched=True)

print("\nSample Formatted Record:")
print(train_dataset[0]["text"][:350], "...")

## 4. Scaled QLoRA Training with Hugging Face TRL SFTTrainer
We configure modern `SFTConfig` and `processing_class=tokenizer` with:
- 8-bit Paged AdamW (`paged_adamw_8bit`) to prevent CUDA OOM
- Mixed precision (`bf16`/`fp16`)
- Cosine learning rate decay with 5% warmup
- Periodic adapter checkpointing to prevent progress loss

In [ ]:
import os
import torch
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "models/qwen2.5_stance_qlora_full"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Configure modern SFTConfig from TRL
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,  # Adjust epochs as desired (e.g. 3, 5, or 10)
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    fp16=not torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    bf16=torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    report_to="none",
    gradient_checkpointing=True,
    dataset_text_field="text",
    max_length=512,
)

# Safely inject warmup_ratio
setattr(sft_config, "warmup_ratio", 0.05)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=sft_config,
)

print("==> Starting Scaled QLoRA Fine-Tuning...")
trainer.train()

# Save final tuned adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"==> Adapter weights saved successfully to {OUTPUT_DIR}")

## 5. Evaluation: General Held-Out Test Set & Hard-Negative Benchmark
We evaluate stance prediction accuracy and F1 scores using uppercase-normalized metric extraction:

In [ ]:
from tqdm import tqdm
from src.qwen_qlora_trainer import parse_stance_prediction, compute_stance_metrics, format_chat_record

model.eval()

def evaluate_dataset(eval_df, sample_limit=250):
    sample_df = eval_df.iloc[:sample_limit].copy()
    y_true = sample_df["stance"].astype(str).str.strip().str.upper().tolist()
    y_pred = []

    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Evaluating"):
        messages = format_chat_record(str(row["conclusion"]), str(row["premise_text"]), stance=None)
        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=30,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        gen_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        pred_stance = parse_stance_prediction(gen_text)
        y_pred.append(pred_stance)

    metrics = compute_stance_metrics(y_true, y_pred)
    return metrics, y_true, y_pred

print("=== EVALUATING ON GENERAL TEST SET ===")
gen_metrics, gen_true, gen_pred = evaluate_dataset(test_df, sample_limit=200)
for k, v in gen_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

if df_hard is not None:
    print("\n=== EVALUATING ON MINED HARD NEGATIVES CHALLENGE SET ===")
    hn_metrics, hn_true, hn_pred = evaluate_dataset(df_hard, sample_limit=len(df_hard))
    for k, v in hn_metrics.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## 6. Live Dialectical Testing Playground
Test real-world debate motions and premises against the full-corpus fine-tuned Qwen model:

In [ ]:
def predict_live(topic: str, premise: str):
    messages = format_chat_record(topic, premise, stance=None)
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    stance = parse_stance_prediction(response)

    print(f"Motion:   {topic}")
    print(f"Premise:  {premise}")
    print(f"Response: {response.strip()}")
    print(f"Parsed:   {stance}")
    print("-" * 50)

# Test 1: Adversarial Quoting
predict_live(
    topic="Nuclear Energy Expansion",
    premise="My opponent claims that nuclear reactors emit zero emissions and provide reliable baseload power. While true on paper, the catastrophic financial burden and unresolved waste storage render expansion irresponsible."
)

# Test 2: Concession Structure
predict_live(
    topic="Universal Basic Income",
    premise="I concede that UBI will increase federal deficits in the initial fiscal quarter. However, the eradication of extreme poverty and stimulation of local commerce vastly outweighs short-term treasury friction."
)

# Test 3: Subtle Refutation
predict_live(
    topic="Artificial Intelligence Regulation",
    premise="Unilateral legislative bans will simply export algorithmic development to jurisdictions with zero safety oversight, leaving our institutions completely unprotected."
)

## 7. Google Drive Checkpoint Export
Persist adapter weights permanently to your Google Drive:

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_dest = "/content/drive/MyDrive/Arg_Me_Checkpoints/qwen_qlora_full"
    !mkdir -p "$drive_dest"
    !cp -r models/qwen2.5_stance_qlora_full/* "$drive_dest/"
    print(f"==> Full-corpus adapter weights exported to: {drive_dest}")
else:
    print("Local adapter checkpoints saved at:", Path(OUTPUT_DIR).resolve())